In [5]:
import pandas as pd

df_model = pd.read_csv("data/insurance_with_predictions.csv")

# final pricing choice
base_margin = 0.10
buffer = 2000

df_model['premium_final'] = df_model['predicted_cost'] * (1 + base_margin) + buffer
df_model['profit_final'] = df_model['premium_final'] - df_model['charges']

# pricing error (over / under pricing)
df_model['pricing_error'] = df_model['premium_final'] - df_model['charges']

# residual (prediction error)
df_model['residual'] = df_model['charges'] - df_model['predicted_cost']

# sanity check
print("Average premium:", df_model['premium_final'].mean())
print("Average profit:", df_model['profit_final'].mean())
print("Loss ratio:", (df_model['profit_final'] < 0).mean())


Average premium: 16658.93456545549
Average profit: 3388.5123003142357
Loss ratio: 0.08520179372197309


## Final pricing rule ##

In this project, the final insurance pricing strategy is defined as:
$\text{Premium}_i = \hat{C}_i \times (1 + m) + k$

where:
- $\hat{C}_i$: predicted medical cost for individual $i$
- $m$: margin (profit loading)
- $k$: safety buffer (fixd adjustment for uncertainty)

### Components of the pricing rule ###
1. Predicted cost ($\hat{C}$)

The base of the pricing model is the predicted medical cost, estimated using a regression model.
- Captures expected healthcare expenses
- Reflects individual risk factors (e.g., age, BMI, smoking status)
- Serves as the primary driver of the premium

2. Margin ($m$)

A proportional markup is applied to ensure profitability.
$\hat{C} \times (1 + m)$
- Accounts for administrative costs and profit
- Scales with the level of predicted risk
- Maintains proportional pricing across individuals

3. Safety Buffer ($k$)

A fixed amount is added to all premiums:
$+k$
- Designed to account for model uncertainty
- Helps mitigate underestimation of high-cost cases
- Provides protection against unexpected medical expenses

### Rationale for the final design ###
Earlier analysis showed that:
- Increasing margins significantly improves profit
- However, it has limited impact on reducing loss cases
- Losses are primarily caused by prediction errors, especially in unexpected high-cist individuals

Therefore, a safety buffer is introduced to address this limitation.

### Interpretation ###
The final pricing rule can be interpreted as:
    Premiums are primarily determined by predicted medical costs, with an additional proportional margin for profitability and a fixed safety buffer to account for model uncertainty.

### Key Insight ###
- Margin controls profitability
- Buffer controls risk from prediction error

Together, they balance:
- Financial performance
- Stability to uncertainty

### Limitations ###
- The safet buffer applies uniformly to all individuals
- This may introduce cross-subsidisation, where low-risk individuals contribute more than their expected cost
- Further analysis is required to evaluate fairness across groups.

In [6]:
sample_cases = df_model[['age', 'bmi', 'smoker', 'predicted_cost', 'premium_final', 'charges', 'profit_final']].sample(5, random_state=42)
print(sample_cases)

      age     bmi smoker  predicted_cost  premium_final      charges  \
764    45  25.175     no     9855.093520   12840.602872   9095.06825   
887    36  30.020     no     7804.873583   10585.360941   5272.17580   
890    64  26.885    yes    28972.725662   33869.998228  29330.98315   
1293   46  25.745     no    10108.859112   13119.745024   9301.89355   
259    19  31.920    yes    34778.257534   40256.083287  33750.29180   

      profit_final  
764    3745.534622  
887    5313.185141  
890    4539.015078  
1293   3817.851474  
259    6505.791487  


## Individual explainability ##

To better understand how the pricing rule operates at the individual level, we examine a sample of customers and decompose their premiums into components.

The pricing rule is defined as:
$\text{Premium} = \hat{C} \times (1 + 0.10) + 2000$
where:
- $\hat{C}$: predicted cost
- 10%: margin
- 2000: safety buffer

### Case 1 ###
* Age: 45, BMI: 25.18, Non-smoker
* Predicted Cost: £9,855
* Premium: £12,841
* Actual Cost: £9,095
* Profit: £3,746

The predicted cost is relatively moderate, reflecting a low-risk profile (non-smoker, normal BMI).
Applying a 10% margin increases the cost slightly, and the additional £2,000 buffer raises the final premium further.

Since the actual medical cost is lower than expected, this case generates a positive profit.

### Case 2 ###
* Age: 36, BMI: 30.02, Non-smoker
* Predicted Cost: £7,805
* Premium: £10,585
* Actual Cost: £5,272
* Profit: £5,313

Although the individual is slightly above the BMI threshold, the predicted cost remains relatively low.
The buffer contributes a significant portion of the premium in this case.

The actual cost is substantially lower than predicted, resulting in a high profit.

### Case 3 ###
* Age: 64, BMI: 26.89, Smoker
* Predicted Cost: £28,973
* Premium: £33,870
* Actual Cost: £29,331
* Profit: £4,539

This individual is classified as high-risk due to smoking status and age.
As a result, the predicted cost is high, which directly leads to a higher premium.

The actual cost is close to the predicted value, indicating that the model performs well for this type of risk profile.

### Case 4 ###
* Age: 46, BMI: 25.75, Non-smoker
* Predicted Cost: £10,109
* Premium: £13,120
* Actual Cost: £9,302
* Profit: £3,818

Similar to Case 1, this individual falls into a relatively low-risk category.
The premium is primarily driven by the predicted cost, with the buffer providing additional protection.

The pricing remains stable and produces a moderate profit.

### Case 5 ###
* Age: 19, BMI: 31.92, Smoker
* Predicted Cost: £34,778
* Premium: £40,256
* Actual Cost: £33,750
* Profit: £6,506

This is a high-risk individual due to both smoking status and high BMI.
The model assigns a very high predicted cost, which leads to a substantially higher premium.

The actual cost is slightly lower than predicted, resulting in a large profit.

### Key Observations ###
From these examples, several patterns emerge:

* Premiums are primarily driven by predicted costs
* High-risk individuals (e.g., smokers) receive significantly higher premiums
* The safety buffer has a larger relative impact on low-cost individuals
* When predictions are accurate, the pricing model produces stable and consistent profits

These individual cases demonstrate that:

The pricing system is largely driven by predicted medical costs, while the margin ensures profitability and the safety buffer provides protection against uncertainty.

In [7]:
# normal case
normal_cases = df_model[['age', 'bmi', 'smoker', 'predicted_cost', 'premium_final', 'charges', 'profit_final']].sample(5, random_state=42)
print("1. Normal cases: ", normal_cases, "\n")

# loss case
loss_cases = df_model[df_model['profit_final'] < 0][['age', 'bmi', 'smoker', 'predicted_cost', 'premium_final', 'charges', 'profit_final']].head(5)
print("2. Loss cases: ", loss_cases, "\n")

# high-profit case
high_profit_cases = df_model[df_model['profit_final'] > 5000][['age', 'bmi', 'smoker', 'predicted_cost', 'premium_final', 'charges', 'profit_final']].head(5)
print("3. High profit cases: ", high_profit_cases, "\n")

1. Normal cases:        age     bmi smoker  predicted_cost  premium_final      charges  \
764    45  25.175     no     9855.093520   12840.602872   9095.06825   
887    36  30.020     no     7804.873583   10585.360941   5272.17580   
890    64  26.885    yes    28972.725662   33869.998228  29330.98315   
1293   46  25.745     no    10108.859112   13119.745024   9301.89355   
259    19  31.920    yes    34778.257534   40256.083287  33750.29180   

      profit_final  
764    3745.534622  
887    5313.185141  
890    4539.015078  
1293   3817.851474  
259    6505.791487   

2. Loss cases:      age     bmi smoker  predicted_cost  premium_final      charges  \
3    33  22.705     no     6726.559686    9399.215655  21984.47061   
9    60  25.840     no    13811.963890   17193.160279  28923.13692   
34   28  36.400    yes    39177.439684   45095.183653  51194.55914   
45   55  37.300     no    12694.127130   15963.539843  20630.28351   
62   64  24.700     no    14892.254133   18381.479547  

## Case-based Explainability ##
To further interpret the pricing mechanism, we analyse three representative groups of individuals:

1. Normal cases (well-estimated scenarios)
2. Loss cases (underestimation scenarios)
3. High-profit cases (overestimation or conservative pricing scenarios)

The pricing rule remains:

$\text{Premium} = \hat{C} \times (1 + 0.10) + 2000$

### 1. Normal cases ###
These cases represent situations where the model performs reasonably well and produces stable profits.

* Predicted Cost: £7,800 – £34,800
* Premium: £10,500 – £40,200
* Actual Cost: Generally close to predicted values
* Profit: £3,700 – £6,500

In these cases, the predicted cost is relatively aligned with the actual medical cost.
The margin ensures a proportional increase, while the safety buffer adds a consistent uplift.

As a result:

* Pricing remains stable
* Profit is consistently positive
* The model behaves as expected

When prediction accuracy is reasonable, the pricing rule produces reliable and consistent profits.

### 2. Loss cases ###
These cases highlight the main weakness of the system.

* Predicted Cost: £6,700 – £39,100
* Premium: £9,300 – £45,000
* Actual Cost: £20,000 – £51,000
* Profit: -£4,600 to -£12,500

In these cases, the predicted cost significantly underestimates the true medical cost.

For example:

* Predicted: £6,700 → Premium: ~£9,400
* Actual: ~£22,000

Even after applying both margin and buffer, the premium remains far below the actual cost.

Therefore, losses are not caused by insufficient pricing, but by severe prediction errors.

Additional Observation

* Many loss cases are non-smokers
* Indicates difficulty in capturing unexpected high-cost events

### 3. High-profit cases ###
These cases represent scenarios where the insurer benefits the most.

* Predicted Cost: £6,300 – £41,400
* Premium: £8,900 – £47,500
* Actual Cost: £3,800 – £39,600
* Profit: £5,100 – £8,000

These cases occur when:

* The predicted cost is relatively high, or
* The actual cost turns out to be lower than expected

In both situations:

* The premium is sufficiently high
* The realised cost is lower
* Resulting in substantial profit

High-risk individuals or overestimated cases generate the majority of profits.

### Cross-case Comparison ###

| Case Type   | Prediction Accuracy        | Profit Outcome   | Main Driver                          |
|------------|---------------------------|------------------|--------------------------------------|
| Normal      | Moderate                  | Stable profit    | Balanced estimation                  |
| Loss        | Poor (underestimation)    | Negative profit  | Prediction error                     |
| High-profit | Conservative / overestimate | High profit      | Overestimation or high-risk pricing  |

### Overall Interpretation ###
From these three case types, a clear pattern emerges:

* The pricing model works well when predictions are accurate
* Profitability is primarily driven by prediction quality
* Losses occur when the model fails to capture extreme costs

### Key Takeaway ###
The effectiveness of the pricing system depends less on the margin or buffer itself, and more on the accuracy and reliability of the underlying cost prediction.

### Then what now? ###
These observations naturally lead to the next question:

If pricing depends heavily on prediction accuracy, is the resulting system fair across different groups?

This motivates the fairness evaluation in the next stage.

In [8]:
# pricing error
df_model['pricing_error'] = df_model['premium_final'] - df_model['charges']

In [9]:
# smoker
print("\n=== Smoker Group Analysis ===")
print("Average profit:")
print(df_model.groupby('smoker')['profit_final'].mean())

print("\nAverage pricing error:")
print(df_model.groupby('smoker')['pricing_error'].mean())

print("\nLoss ratio:")
print(df_model.groupby('smoker')['profit_final'].apply(lambda x: (x < 0).mean()))


=== Smoker Group Analysis ===
Average profit:
smoker
no     2903.304508
yes    5272.676864
Name: profit_final, dtype: float64

Average pricing error:
smoker
no     2903.304508
yes    5272.676864
Name: pricing_error, dtype: float64

Loss ratio:
smoker
no     0.093045
yes    0.054745
Name: profit_final, dtype: float64


## Fairness Analysis - Smoker vs Non-Smoker ##

### Summary Statistics ###
| Group        | Average Profit | Average Pricing Error | Loss Ratio |
|-------------|----------------|----------------------|------------|
| Non-smoker  | £2,903         | £2,903               | 9.30%      |
| Smoker      | £5,273         | £5,273               | 5.47%      |

### 1. Profit Distribution ###
Smokers generate significantly higher profits compared to non-smokers.
* Smokers: ~£5,273
* Non-smokers: ~£2,903

This indicates that "high-risk individuals (smokers) are charged substantially higher premiums, resulting in greater profitability for the insurer."

### 2. Pricing Error ###
The pricing error is also much higher for smokers: Smokers pay ~£5,273 more than their actual cost on average, whereas non-smokers pay ~£2,903 more.
This suggests that smokers are consistently overcharged relative to their realised medical costs.

### 3. Loss Ratio ###
* Non-smokers: 9.30%
* Smokers: 5.47%

This shows that losses occur more frequently among non-smokers.

### Key Insights ###
* **Profit Concentration**: A large proportion of total profit is generated from smokers.
* **Risk-Based Pricing Effect**: The pricing model charges high-risk individuals more, which is economically rational but may raise fairness concerns.
* **Prediction Limitation**: Non-smokers have a higher loss ratio, suggesting that unexpecting high-cost cases are not well captured by the model.

### Fairness Implications ###
This result highlights 2 competing perspectives:
* Economic Fairness (Risk-Based): Charging smokers more is consistent with higher expected risk and this aligns with traditional insurance pricing principles.
* Equity Concerns: Smokers pay significantly more than their actual cost on average. Non-smokers indirectly benefit from this structure through lower pricing relative to risk.

### Key Takeaway ###
The pricing model is economically efficient but leads to uneven burden distribution, where high-risk individuals (smokers) contribute disproportionately to overall profit.

In [ ]:
# bmi
print("\n=== BMI Group Analysis ===")
print("Average profit:")
print(df_model.groupby('bmi_30')['profit_final'].mean())

print("\nAverage pricing error:")
print(df_model.groupby('bmi_30')['pricing_error'].mean())

print("\nLoss ratio:")
print(df_model.groupby('bmi_30')['profit_final'].apply(lambda x: (x < 0).mean()))

In [ ]:
# low risk vs high risk
df_model['risk_group'] = df_model['predicted_cost'].apply(lambda x: 'high' if x > df_model['predicted_cost'].median() else 'low')

print("\n=== Risk Group Analysis ===")
print(df_model.groupby('risk_group')['profit_final'].mean())
print(df_model.groupby('risk_group')['pricing_error'].mean())